# Run 4 on Kaggle — line chart toward 100 %: Qwen3.5-9B + LoRA on Quest videos (free T4)

**Before running:** right panel → *Accelerator* = **GPU T4** (×1 or ×2, only one is used), *Internet* = **On**,
*Add Input* = your dataset with `dxr_video_upload.zip` (and, optional, the folder `lora_qwen35_line` from Google Drive).
Then **Save Version → Save & Run All (Commit)**: it runs in the background, you can close the tab.
Results appear under the version's **Output** tab: `video_results_line_r4.zip` and `RESULTS_line.md`.

| Step | What it does | T4 time |
|---|---|---|
| 1 | install Unsloth | ~6 min |
| 2–4 | find your files, add the updated code, settings | 1 min |
| 5 | **quick result:** your run-3 model (97.9 %) + 7-frame median votes + offset fix — only if you added `lora_qwen35_line` | ~45 min |
| 6 | 1,500 synthetic line clips with finer gridlines (strict: made-up countries and values) | ~5 min |
| 7 | baseline: untrained Qwen3.5-9B on your video | ~10 min |
| 8 | train run 4 | ~3–3.5 h |
| 9 | test run 4: whole video, median votes, offset fix | ~45 min |
| 10 | results vs ground truth + every point outside 5 % | seconds |

**What the model learns from:** only synthetic charts this notebook draws (made-up countries, made-up values, each with
its exact table). Your Quest video is used only as the test, and your truth table only by the scorer after the model
has answered. The offset correction is learnt from synthetic clips, never from the truth table.

In [ ]:
# 1 · Install
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth uses one GPU
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q --upgrade --no-deps "transformers==5.2.0" "trl==0.22.2" "tokenizers>=0.22.0,<=0.23.0"
!python3 -c "import unsloth, transformers; print('INSTALL OK - unsloth ready, transformers', transformers.__version__)"

In [ ]:
# 2 · Find your files in /kaggle/input and set up the project folder
import os, glob, zipfile, shutil, json, csv
WORK = "/kaggle/working/dxr"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{WORK}/real_clips/real.jsonl"):
    found = glob.glob("/kaggle/input/**/real_clips/real.jsonl", recursive=True)
    if found:                                          # Kaggle unzipped the upload for us
        src = os.path.dirname(os.path.dirname(found[0]))
        for n in os.listdir(src):
            s = os.path.join(src, n)
            (shutil.copytree(s, f"{WORK}/{n}", dirs_exist_ok=True) if os.path.isdir(s) else shutil.copy(s, WORK))
    else:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        assert zips, "No project found: add your dataset with dxr_video_upload.zip (Add Input, right panel)"
        zipfile.ZipFile(zips[0]).extractall(WORK)
RUN3 = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)), None)
os.chdir(WORK)
print("project:", WORK, "| run-3 adapter:", RUN3 or "not added (step 5 will be skipped)")
!ls
!grep line real_clips/real.jsonl | cut -c1-140
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3 · Updated code (max-length fix, scorer fallback, finer gridlines, 7-frame median votes, offset correction)
import base64, io, tarfile
os.chdir(WORK)
CODE = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode="r:xz").extractall(WORK)
!python3 vlm_eval/run_local_vlm.py

In [ ]:
# 4 · Settings (T4 16 GB: 4-bit, 640 px)
import torch
os.chdir(WORK)
CHART, VIDEO = "line", "line_clear"
MODEL = "unsloth/Qwen3.5-9B"
TRAIN, STEPS, SIDE, MAXLEN, X = 1500, 188, 640, 6144, "--load-4bit"
VALN = 5                                    # synthetic clips used to learn the offset (each costs 9 answers)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gb > 30:                                 # if Kaggle ever gives a bigger GPU
    TRAIN, STEPS, MAXLEN, X, VALN = 2000, 250, 8192, "", 10
ADAPTER, RUN = f"lora_qwen35_{CHART}_r4", f"qwen35_r4_{CHART}"
DATA, REAL = f"finetune_video/data_{CHART}{SIDE}", "real_clips/real.jsonl"
def done(model, inp):
    return os.path.exists("results_video.csv") and any(
        r["model"] == model and r["set"] == "real" and r["clip"] == VIDEO and r["input"] == inp for r in csv.DictReader(open("results_video.csv")))
print(f"GPU {gb:.0f} GB | {CHART} | {VIDEO} | {TRAIN} clips, {STEPS} steps, {SIDE} px | {X or '16-bit'}")

In [ ]:
# 5 · Quick result: run-3 model (trained on the A100) + median votes + offset fix  [only if you added lora_qwen35_line]
os.chdir(WORK)
if RUN3 and not done("qwen35_r3_line", "video_cal"):
    if not os.path.exists("finetune_video/data_line640/val.jsonl"):
        !python finetune_video/make_video_dataset.py --strict --charts line --train 8 --val 30 --out finetune_video/data_line640
    !python finetune_video/eval_median.py --model {RUN3} --name qwen35_r3_line --charts {CHART} --real {REAL} --videos {VIDEO} --val {DATA}/val.jsonl --val-n {VALN} --max-side {SIDE} {X}
else:
    print("skipped" if not RUN3 else "already done")

In [ ]:
# 6 · Synthetic training clips (strict) + a look at one next to your video
from PIL import Image
from IPython.display import display, Markdown
os.chdir(WORK)
if not os.path.exists(f"{DATA}/train.jsonl") or sum(1 for _ in open(f"{DATA}/train.jsonl")) < TRAIN:
    !python finetune_video/make_video_dataset.py --strict --charts {CHART} --train {TRAIN} --val 30 --size {SIDE}x{SIDE*3//4} --out {DATA}
def strip(paths, base):
    ims = [Image.open(f"{base}/{p}").resize((240, 180)) for p in paths]
    s = Image.new("RGB", (240 * len(ims), 180), "white")
    for i, im in enumerate(ims): s.paste(im, (240 * i, 0))
    return s
t = json.loads(open(f"{DATA}/train.jsonl").readline()); print("synthetic training clip (made-up data)"); display(strip(t["frames"], DATA))
for l in open(REAL):
    r = json.loads(l)
    if r["video"] == VIDEO: print("your Quest video (test only):", VIDEO); display(strip(r["frames"], "real_clips"))

In [ ]:
# 7 · Baseline: untrained Qwen3.5-9B on the same video
os.chdir(WORK)
if not done("qwen35_base", "video"):
    !python finetune_video/eval_video.py --model {MODEL} --name qwen35_base --charts {CHART} --real {REAL} --videos {VIDEO} --val {DATA}/val.jsonl --val-n 5 --max-side {SIDE} {X}
else:
    print("already done")

In [ ]:
# 8 · Train run 4 (synthetic clips only)
os.chdir(WORK)
if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    !python finetune_video/train_video_lora.py --base {MODEL} --data {DATA} --steps {STEPS} --max-side {SIDE} --max-length {MAXLEN} --out {ADAPTER} {X}
print(open(f"{ADAPTER}/training_info.json").read())

In [ ]:
# 9 · Test run 4: whole video, median of 7-frame votes, offset learnt on synthetic clips only
os.chdir(WORK)
if not done(RUN, "video_cal"):
    !python finetune_video/eval_median.py --model {ADAPTER} --name {RUN} --charts {CHART} --real {REAL} --videos {VIDEO} --val {DATA}/val.jsonl --val-n {VALN} --max-side {SIDE} {X}
else:
    print("already done")

In [ ]:
# 10 · Results vs ground truth (also saved as RESULTS_line.md and video_results_line_r4.zip in Output)
import importlib.util, statistics
os.chdir(WORK)
spec = importlib.util.spec_from_file_location("ev", "finetune_video/eval_video.py"); ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)
st = ev.st
gt = st.read(os.path.join(st.GT, st.FILES[CHART]))
tv = {(st.norm(g["Country"]), int(float(g["Year"]))): float(g["Life_expectancy"]) for g in gt}
def pv_of(path):
    pred = st.read(path) if os.path.exists(path) else []
    return {(st.norm(p.get("Country", "")), int(float(p["Year"]))): float(p["Life_expectancy"])
            for p in pred if st.num(p.get("Year")) is not None and st.num(p.get("Life_expectancy")) is not None}
rows = [r for r in csv.DictReader(open("results_video.csv")) if r["chart"] == CHART]
syn = {}
for r in rows:
    if r["set"] == "synthetic": syn.setdefault((r["model"], r["input"]), []).append(float(r["value_acc"] or 0))
names = {"video": "whole video", "median": "median of 7-frame votes", "median_cal": "median + offset fix", "video_cal": "whole video + offset fix"}
lines = ["| Model | Method | Within 5 % | Within 2 % | Mean error | Synthetic within 5 % |", "|---|---|---|---|---|---|"]
best = None
for r in [r for r in rows if r["set"] == "real" and r["clip"] == VIDEO]:
    pv = pv_of(r["prediction"]); acc = float(r["value_acc"] or 0)
    w2 = 100 * sum(k in pv and abs(pv[k] - v) <= 0.02 * v for k, v in tv.items()) / len(tv)
    s = syn.get((r["model"], "median" if "median" in r["input"] else "video"))
    lines.append(f"| {r['model']} | {names.get(r['input'], r['input'])} | **{100 * acc:.1f} %** | {w2:.1f} % | {100 * float(r['value_mre'] or 0):.2f} % | {f'{100 * statistics.mean(s):.0f} %' if s else '–'} |")
    if best is None or acc > best[0]: best = (acc, r)
out = f"### {CHART} · real Quest video `{VIDEO}` ({len(tv)} true values)\n\n" + "\n".join(lines) + "\n\n"
if best:
    pv = pv_of(best[1]["prediction"])
    miss = ["| Country | Year | Truth | Model | Error |", "|---|---|---|---|---|"]
    for g in gt:
        k = (st.norm(g["Country"]), int(float(g["Year"]))); v = float(g["Life_expectancy"]); p = pv.get(k)
        if p is None or abs(p - v) > 0.05 * v:
            miss.append(f"| {g['Country']} | {g['Year']} | {v:.1f} | {p if p is not None else 'missing'} | {'' if p is None else f'{100 * (p - v) / v:+.1f} %'} |")
    out += f"**Best: {best[1]['model']} · {names.get(best[1]['input'], best[1]['input'])} = {100 * best[0]:.1f} %.** "
    out += "Every point is within 5 %." if len(miss) == 2 else "Points outside 5 %:\n\n" + "\n".join(miss)
open(f"/kaggle/working/RESULTS_{CHART}.md", "w").write(out)
display(Markdown(out))
!python finetune_video/eval_video.py --compare --charts {CHART} --out compare_{CHART}.md > /dev/null
!cd {WORK} && zip -qr /kaggle/working/video_results_{CHART}_r4.zip results_video.csv compare_{CHART}.md predictions {ADAPTER}/training_info.json
!rm -rf {WORK}/finetune_video/data_* {WORK}/unsloth_compiled_cache {WORK}/outputs_video   # keep the Output small
print("Output: RESULTS_line.md, video_results_line_r4.zip, dxr/lora_qwen35_line_r4 (the trained adapter)")

### If something goes wrong
* **`INSTALL OK` missing:** send the last 20 lines of step 1.
* **"No project found":** right panel → *Add Input* → your dataset with `dxr_video_upload.zip`.
* **Out of memory in step 8:** in step 4 set `SIDE, MAXLEN = 512, 4096` and run again.
* **Next chart:** in step 4 set `CHART, VIDEO = "pie", "pie_clear"` (step 10's point table is line-only).